# From Modular Forms to Generation Counting — Technical Notebook

**Phase 5b, Waves 4-5** | SK-EFT Hawking Project

Derives the SM generation constraint $N_f \equiv 0 \pmod{3}$ from the Dedekind eta function's modular transformation, with the coefficient $c_- = 8$ derived from SM fermion content.

**Lean modules:** WangBridge.lean (9 thms), ModularInvarianceConstraint.lean (12 thms), GenerationConstraint.lean (13 thms)

**Key result:** $24/8 = 3$ — the numerator is math (Dedekind eta), the denominator is physics (SM fermions).

In [ ]:
# viz-ref: fig_sm_generation_constraint
# viz-ref: fig_modular_invariance_phase
from src.core.formulas import (
    wang_bridge_central_charge,
    modular_t_phase,
    dedekind_eta_origin_of_24,
    sm_generation_constraint,
)
from src.core.constants import SM_FERMION_DATA
from src.core.visualizations import (
    fig_sm_generation_constraint,
    fig_modular_invariance_phase,
    COLORS,
)

## 1. Wang Bridge: Deriving $c_- = 8 N_f$

The SM has 16 Weyl fermion components per generation (with $\nu_R$). Each contributes $c = 1/2$ to the chiral central charge.

In [ ]:
# Step 1: Count SM Weyl fermions
total_weyl = sum(f['components'] for f in SM_FERMION_DATA.values())
print(f'SM Weyl fermions per generation: {total_weyl}')
for name, f in SM_FERMION_DATA.items():
    print(f'  {name}: {f["components"]} components')

In [ ]:
# Step 2: Derive central charge
with_nu_R = wang_bridge_central_charge(16)
without_nu_R = wang_bridge_central_charge(15)

print(f'With ν_R:    c₋ = {with_nu_R["c_minus_exact"]} = {with_nu_R["c_minus_per_gen"]} (integral: {with_nu_R["is_integral"]})')
print(f'Without ν_R: c₋ = {without_nu_R["c_minus_exact"]} = {without_nu_R["c_minus_per_gen"]} (integral: {without_nu_R["is_integral"]})')
print(f'\nν_R required for integral central charge: {without_nu_R["requires_nu_R"]}')

## 2. Framing Anomaly: The T-Phase

The Dedekind eta function $\eta(\tau) = q^{1/24} \prod(1-q^n)$ transforms as $\eta(\tau+1) = e^{2\pi i/24} \eta(\tau)$. The partition function picks up phase $e^{2\pi i c_-/24}$ under $T: \tau \to \tau+1$.

In [ ]:
# Compute T-phase for N_f = 1..12
print(f'{"N_f":>5} {"c₋":>5} {"c₋ mod 24":>10} {"Phase":>20} {"Invariant":>10} {"3|N_f":>6}')
print('-' * 60)
for n in range(1, 13):
    r = modular_t_phase(8 * n)
    g = sm_generation_constraint(n)
    phase_str = f'{r["phase"].real:+.3f}{r["phase"].imag:+.3f}i'
    print(f'{n:>5} {8*n:>5} {r["c_minus_mod_24"]:>10} {phase_str:>20} {str(r["is_modular_invariant"]):>10} {str(g["satisfies_generation_constraint"]):>6}')

In [ ]:
# Visualize the T-phase on the unit circle
fig_modular_invariance_phase().show()

## 3. The Origin of 24

The number 24 in $c_- \equiv 0 \pmod{24}$ comes from the $q^{1/24}$ prefactor in the Dedekind eta function, which traces to $\zeta(-1) = -1/12$ (zeta regularization).

In [ ]:
info = dedekind_eta_origin_of_24()
print(f'η(τ) prefactor exponent: {info["eta_prefactor_exponent"]}')
print(f'ζ(-1) = {info["zeta_minus_one"]} (Ramanujan summation)')
print(f'Casimir energy: {info["casimir_energy_formula"]}')
print(f'T-phase: {info["T_phase"]}')
print(f'\n24 = 8 × 3:')
for k, v in info['factorization_24'].items():
    print(f'  {k}: {v}')

In [ ]:
# Generation constraint visualization
fig_sm_generation_constraint().show()

## 4. The Complete Chain

| Step | Content | Lean Module |
|------|---------|-------------|
| 1 | SM has 16 Weyl per generation | SMFermionData.lean |
| 2 | $c_- = 16/2 = 8$ per generation | WangBridge.lean |
| 3 | $\eta(\tau+1) = \zeta_{24} \eta(\tau)$ → $24 \mid c_-$ | ModularInvarianceConstraint.lean |
| 4 | $24 \mid 8N_f \Rightarrow 3 \mid N_f$ | GenerationConstraint.lean |
| 5 | $N_f = 3$ minimal nontrivial | GenerationConstraint.lean |

## 5. Algebraic Core: Ext Computation over A(1)

The generation constraint chain traces through Rokhlin's theorem (16 | σ for spin 4-manifolds).
The algebraic content of this "16" is now machine-checked: the minimal free resolution of F₂
over A(1) = ⟨Sq¹, Sq²⟩ (the 8-dimensional Steenrod subalgebra) has been verified end-to-end
in Lean 4 via native_decide.

**Key correction:** Ext⁴ has dimension 3 (not 16). The number 16 enters through the
infinite h₀-tower in stem 4, which assembles to ℤ via Adams spectral sequence extensions.

This is the **first machine-checked Ext computation over any Steenrod subalgebra** in any proof assistant.

In [ ]:
# viz-ref: fig_ext_chart
from src.core.visualizations import fig_ext_chart
from src.core.formulas import a1_ext_dimension, a1_ext_generator_bidegrees, bordism_hypothesis_count

# Ext dimensions from the machine-checked resolution
print("Ext dimensions (machine-checked in A1Ext.lean):")
for n in range(6):
    dim = a1_ext_dimension(n)
    gens = a1_ext_generator_bidegrees(n)
    stems = [t - s for s, t in gens]
    print(f"  Ext^{n} = F2^{dim}  (generators at stems {stems})")

total = sum(a1_ext_dimension(n) for n in range(6))
print(f"\nTotal basis elements through degree 5: {total}")

fig_ext_chart().show()


In [ ]:
# viz-ref: fig_a1_resolution_structure
from src.core.visualizations import fig_a1_resolution_structure
from src.core.constants import A1_RESOLUTION_RANKS, A1_EXT_GENERATORS

print("Resolution ranks (= Ext dimensions by minimality):")
print(f"  P0..P5: {A1_RESOLUTION_RANKS}")
print()
print("Ext algebra generators:")
for name, data in A1_EXT_GENERATORS.items():
    s, t = data["bidegree"]
    print(f"  {name}: bidegree ({s},{t}), stem {data['stem']}: {data['desc']}")

fig_a1_resolution_structure().show()


## 6. Hypothesis Decomposition

The single opaque spin bordism hypothesis (Ω^Spin_4 ≅ ℤ) is now decomposed into
3 focused topological hypotheses, each a standard textbook result:

| Hypothesis | Content | Eliminability | Reference |
|---|---|---|---|
| H1 | H\*(ko; F₂) ≅ A ⊗\_{A(1)} F₂ | Topological | Adams 1974, Ch. 16 |
| H2 | Change-of-rings: Ext\_A ≅ Ext\_{A(1)} | **DISCHARGED** | ChangeOfRings.lean |
| H3 | ASS for ko collapses at E₂ | Topological | Ravenel 2003, Ch. 3 |
| H4 | ABP splitting: π\_n(MSpin) ≅ π\_n(ko) for n < 8 | Topological | ABP 1967 |

H2 was discharged via the Hom-tensor adjunction (pure algebra, no topology needed).
The remaining 3 hypotheses are irreducibly topological — they cannot be proved
without spectrum theory and algebraic topology infrastructure not in any proof assistant.

In [ ]:
# Hypothesis inventory
status = bordism_hypothesis_count()
print("Generation constraint formal status:")
print(f"  Machine-checked components: {status['machine_checked_components']}")
print(f"  Topological hypotheses remaining: {status['topological']}")
print(f"  Hypotheses discharged: {status['discharged']}")
print(f"  New sorry introduced: {status['sorry_introduced']}")
print(f"  New axioms introduced: {status['axioms_introduced']}")
print()
print("This is the strongest formal position achievable")
print("without formalizing algebraic topology in Lean.")
